In [97]:
import stanza

In [85]:
import pandas as pd

In [99]:
nlp = stanza.Pipeline('en', processors='tokenize,pos,lemma,depparse')

2026-11-24 15:11:11 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-11-24 15:11:12 INFO: Downloaded file to C:\Users\erict\AppData\Local\StanfordNLP\stanza\Cache\1.15.0\resources\resources.json
2026-11-24 15:11:12 WARNING: Language en package default expects mwt, which has been added
2026-11-24 15:11:12 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined_nocharlm |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

2026-11-24 15:11:12 INFO: Using device: cpu
2026-11-24 15:11:12 INFO: Loading: tokenize
2026-11-24 15:11:12 INFO: Loading: mwt
2026-11-24 15:11:12 INFO: Loading: pos
2026-11-24 15:11:14 INFO: Loading: lemma
2026-11-24 15:11:15 INFO: Loading: depparse
2026-11-24 15:11:15 INFO: Done loading processors!


In [43]:
url = "https://huggingface.co/datasets/shuttie/dadjokes/resolve/main/train.csv"

In [45]:
df = pd.read_csv(url)

In [49]:
df

,question,response
0,I asked my priest how he gets holy water,"He said it’s just regular water, he just boils..."
1,Life Hack: If you play My Chemical Romance lou...,your grass will cut itself
2,OMG. SISTERS. JAMES. CHARLES. IS. DOING. A,GIVEAWAY his career
3,Why did Mr. Potato Head get pulled over,He was baked
4,On zombie cravings. My kids and i had some fu...,Blood
...,...,...
51995,What's the difference between in-laws and outlaws,Outlaws are wanted
51996,My wife has this weird OCD where she arranges ...,It is an extremely rare dish order
51997,I countered a customer's dadjoke with another ...,"Me: Normally I would, but inflation has just b..."
51998,My first post ever on reddit,My 3 favorite things are eating my family and ...


In [53]:
df['question']

0                 I asked my priest how he gets holy water
1        Life Hack: If you play My Chemical Romance lou...
2               OMG. SISTERS. JAMES. CHARLES. IS. DOING. A
3                 Why did Mr.  Potato Head get pulled over
4        On zombie cravings.  My kids and i had some fu...
                               ...                        
51995    What's the difference between in-laws and outlaws
51996    My wife has this weird OCD where she arranges ...
51997    I countered a customer's dadjoke with another ...
51998                         My first post ever on reddit
51999                                       The bible says
Name: question, Length: 52000, dtype: str

In [91]:
wdt_questions = df[df['question'].str.lower().str.startswith('why did the ', na=False)]['question']
print(f"Candidates found: \n{wdt_questions}")

Candidates found: 
116             Why did the captain hate his boat's anchor
204                     Why did the cowboy get a dachshund
220                 Why did the two punsters get a divorce
407                      Why did the coffee get in trouble
443                   Why did the can crusher quit his job
                               ...                        
51704    Why did the lettuce blush.  Because it saw the...
51716           Why did the moon hide itself when it cried
51750                             Why did the tomato blush
51841    Why did the banana always stare at his reflection
51867          Why did the protractor marry the calculator
Name: question, Length: 844, dtype: str


In [159]:
#find all "Why did x verb y" questions, ignoring copular constructions and near-misses
def find_direct_match(text):
    #print(text)
    questions = nlp(text).sentences
    #print(questions)
    for q in questions:
        words = q.words
        verbs = [w for w in words if w.upos == "VERB"]
        #print(verbs)
        for v in verbs:
            nsubj = [x for x in words if x.head == v.id and x.deprel in ("nsubj", "nsubj:pass")]
            obj = [y for y in words if y.head == v.id and y.deprel == "obj"]
            if nsubj and obj:
                x_tok = nsubj[0]
                y_tok = obj[0]
                return {
              "raw_text": text,
              "x": x_tok.text,
              "x_lemma": x_tok.lemma,
              "verb": v.text,
              "verb_lemma": v.lemma,
              "y": y_tok.text,
              "y_lemma": y_tok.lemma,
          }
    return None
parsed_jokes = []
for entry in wdt_questions: 
    parsed_q = find_direct_match(entry)
    if (parsed_q):
        parsed_jokes.append(parsed_q)
print(f"Direct \"Why did x verb y\" questions:\n{len(parsed_jokes)}")

Direct "Why did x verb y" questions:
433
